# Нейронные сети и глубокое обучение, МНАД ВШЭ

## Домашнее задание 4. Трансформеры. 

### Общая информация

### Оценивание и штрафы

Максимально допустимая оценка за работу без бонусов — 10 баллов. Сдавать задание после указанного срока жесткого дедлайна нельзя.

Сдача работы после мягкого дедлайна штрафуется ступенчато, -1 балл в сутки. Один раз за модуль студентам предоставляется возможность использовать отсрочку и сдать в жесткий дедлайн без штрафа.

Задание выполняется самостоятельно. «Похожие» решения считаются плагиатом и все задействованные студенты (в том числе те, у кого списали) не могут получить за него больше 0 баллов. Если вы нашли решение какого-то из заданий (или его часть) в открытом источнике, необходимо указать ссылку на этот источник в отдельном блоке в конце вашей работы (скорее всего вы будете не единственным, кто это нашел, поэтому чтобы исключить подозрение в плагиате, необходима ссылка на источник).

Неэффективная реализация кода может негативно отразиться на оценке. Также оценка может быть снижена за плохо читаемый код и плохо оформленные графики. Все ответы должны сопровождаться кодом или комментариями о том, как они были получены.

Использование генеративных моделей допустимо на следующих условиях:
- Количество кода, написанное генеративными моделями, не превышает 30%
- Указана модель, использованная для генерации, а также промпт
- В конце работы необходимо описать свой опыт использования генеративного ИИ для решения данного домашнего задания. Укажите как часто Вам приходилось исправлять код своими руками или просить модель что-то исправить. Было ли это быстрее, чем написать код самим? 

В случае невыполнения этих требований работа не оценивается и оценка за неё не превышает 0 баллов.

### О задании

В этой домашней работе вам предстоит добавить к BERT'у декодерную часть и решить задачу написания tl;dr для текстов новостей на русском языке.

Дополнительно к этому на отличную оценку потребуется реализовать менее жадную стратегию выбора следующего токена для генерации.

In [ ]:
!pip install transformers datasets evaluate

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, BertModel, BertTokenizer

## Подготовка данных (0.5 балла)

Мы воспользуемся датасетом с 🤗 Ильи Гусева "gazeta". Он представляет собой пары (полный текст новости -- его саммари). 

Более подробно про датасет можно прочитать [здесь](https://huggingface.co/datasets/IlyaGusev/gazeta)



In [ ]:
# Загрузим данные с попощью библиотеки библиотеки datasets
# Вы вольны взять меньше или больше данных, но что-то около адекватное получается обычно только на >=10%

from datasets import load_dataset

dataset = load_dataset("IlyaGusev/gazeta", split="train[:10%]")

Вы должны помнить, что тексты перед подачей в модель необходимо **токенизировать**.

Добавьте паддинг до `max_length=512` для обучающих данных, а также до `max_length=128` для меток.

Используйте обрезку текстов, длина которых в токенах превышает `max_length`

In [ ]:
# Подготовим данные для модели Bert

model_name = "deepvk/bert-base-uncased"  # Указание модели BERT

tokenizer = AutoTokenizer.from_pretrained(model_name)


def preprocess(examples, use_padding=True):

    pad = "max_length" if use_padding else False

    model_inputs = tokenizer(
        examples["text"], max_length=512, truncation=True, padding=pad
    )

    labels = tokenizer(
        examples["summary"], max_length=128, truncation=True, padding=pad
    )["input_ids"]

    # сдвиг для teacher forcing
    decoder_input_ids = [tokenizer.cls_token_id] + labels[:-1]

    # паддинги не считаем в лоссе
    labels = [x if x != tokenizer.pad_token_id else -100 for x in labels]

    model_inputs["decoder_input_ids"] = decoder_input_ids
    model_inputs["labels"] = labels

    return model_inputs


In [ ]:
tokenized_dataset = dataset.map(preprocess, batched=False)
tokenized_dataset.set_format(
    "torch", columns=["input_ids", "attention_mask", "decoder_input_ids", "labels"]
)


Map:   0%|          | 0/3048 [00:00<?, ? examples/s]

In [ ]:
from torch.utils.data import DataLoader

split_ds = tokenized_dataset.train_test_split(test_size=0.1, seed=42)

split_ds["train"].set_format(
    "torch", columns=["input_ids", "attention_mask", "decoder_input_ids", "labels"]
)
split_ds["test"].set_format(
    "torch", columns=["input_ids", "attention_mask", "decoder_input_ids", "labels"]
)

train_dataloader = DataLoader(split_ds["train"], batch_size=8, shuffle=True)
eval_dataloader = DataLoader(split_ds["test"], batch_size=8)


## Реализация Decoder-cети (3 балла)

В данном разделе вам необходимо **реализовать собственный декодер для генерации текста**.

Можете вдохновляться кодом с семинара. В инициализации весов стоит (но необязательно) вспомнить нюансы.

In [ ]:
import torch
import torch.nn as nn
from transformers import BertModel


class BertSummarizer(nn.Module):
    def __init__(
        self,
        bert_model_name="bert-base-uncased",
        hidden_size=768,
        num_decoder_layers=3,
        num_heads=8,
        dropout=0.1,
    ):
        super().__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        vocab = self.bert.config.vocab_size

        self.embedding = nn.Embedding(vocab, hidden_size)
        self.pos_embedding = nn.Embedding(512, hidden_size)

        layer = nn.TransformerDecoderLayer(
            d_model=hidden_size, nhead=num_heads, dropout=dropout
        )
        self.decoder = nn.TransformerDecoder(layer, num_layers=num_decoder_layers)

        self.fc_out = nn.Linear(hidden_size, vocab)
        self.dropout = nn.Dropout(dropout)

    def generate_square_subsequent_mask(self, T):
        mask = torch.triu(torch.ones(T, T), diagonal=1)
        return mask.masked_fill(mask == 1, float("-inf"))

    def forward(self, input_ids, attention_mask, decoder_input_ids):
        memory = self.bert(
            input_ids=input_ids, attention_mask=attention_mask
        ).last_hidden_state
        memory = memory.transpose(0, 1)

        x = self.embedding(decoder_input_ids)
        pos = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        x = self.dropout(x + self.pos_embedding(pos)).transpose(0, 1)

        tgt_mask = self.generate_square_subsequent_mask(x.size(0)).to(x.device)
        mem_pad_mask = attention_mask == 0

        out = self.decoder(
            tgt=x, memory=memory, tgt_mask=tgt_mask, memory_key_padding_mask=mem_pad_mask
        )
        logits = self.fc_out(out.transpose(0, 1))
        return logits

    def generate(
        self,
        input_ids,
        attention_mask,
        tokenizer,
        max_len=50,
        strategy="greedy",
        top_k=30,
        top_p=0.9,
        num_beams=3,
        temperature=1.0,
    ):
        self.eval()
        memory = self.bert(
            input_ids=input_ids, attention_mask=attention_mask
        ).last_hidden_state
        memory = memory.transpose(0, 1)

        if strategy == "beam":
            return self._beam_search(
                input_ids[:1], attention_mask[:1], tokenizer, memory[:, :1], max_len, num_beams
            )

        dec = torch.full(
            (input_ids.size(0), 1), tokenizer.cls_token_id, dtype=torch.long
        ).to(input_ids.device)

        eos_id = tokenizer.sep_token_id

        for _ in range(max_len):
            x = self.embedding(dec)
            pos = torch.arange(x.size(1), device=x.device).unsqueeze(0)
            x = (x + self.pos_embedding(pos)).transpose(0, 1)

            tgt_mask = self.generate_square_subsequent_mask(x.size(0)).to(x.device)
            mem_pad_mask = attention_mask == 0

            out = self.decoder(
                tgt=x, memory=memory, tgt_mask=tgt_mask, memory_key_padding_mask=mem_pad_mask
            )
            logits = self.fc_out(out.transpose(0, 1))[:, -1, :]
            logits = logits / max(temperature, 1e-6)
            probs = torch.softmax(logits, dim=-1)

            if strategy == "top_k":
                k = min(top_k, probs.size(-1))
                p, idx = torch.topk(probs, k=k, dim=-1)
                p = p / p.sum(dim=-1, keepdim=True)
                next_id = idx.gather(1, torch.multinomial(p, 1)).squeeze(1)

            elif strategy == "top_p":
                p, idx = torch.sort(probs, descending=True, dim=-1)
                cum = torch.cumsum(p, dim=-1)

                keep = cum <= top_p
                keep[:, 0] = True

                p = p * keep
                p = p / p.sum(dim=-1, keepdim=True)
                next_id = idx.gather(1, torch.multinomial(p, 1)).squeeze(1)

            else:
                next_id = probs.argmax(dim=-1)

            dec = torch.cat([dec, next_id.unsqueeze(1)], dim=1)

            if eos_id is not None and (next_id == eos_id).all():
                break

        return tokenizer.decode(dec[0].tolist(), skip_special_tokens=True)

    def _beam_search(self, input_ids, attention_mask, tokenizer, memory, max_len, num_beams):
        device = input_ids.device
        eos_id = tokenizer.sep_token_id

        beams = [(torch.tensor([[tokenizer.cls_token_id]], device=device), 0.0)]

        for _ in range(max_len):
            new_beams = []

            for seq, score in beams:
                if eos_id is not None and seq[0, -1].item() == eos_id:
                    new_beams.append((seq, score))
                    continue

                x = self.embedding(seq)
                pos = torch.arange(x.size(1), device=device).unsqueeze(0)
                x = (x + self.pos_embedding(pos)).transpose(0, 1)

                tgt_mask = self.generate_square_subsequent_mask(x.size(0)).to(device)
                mem_pad_mask = attention_mask == 0

                out = self.decoder(
                    tgt=x,
                    memory=memory,
                    tgt_mask=tgt_mask,
                    memory_key_padding_mask=mem_pad_mask,
                )
                logits = self.fc_out(out.transpose(0, 1))[:, -1, :]
                logp = torch.log_softmax(logits, dim=-1)

                top_logp, top_idx = torch.topk(logp, k=num_beams, dim=-1)

                for lp, idx in zip(top_logp[0], top_idx[0]):
                    new_seq = torch.cat([seq, idx.view(1, 1)], dim=1)
                    new_beams.append((new_seq, score + lp.item()))

            beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:num_beams]

            if eos_id is not None and all(b[0][0, -1].item() == eos_id for b in beams):
                break

        best_seq = beams[0][0]
        return tokenizer.decode(best_seq[0].tolist(), skip_special_tokens=True)


In [ ]:
# Инициализируем нашу модель и посморим на ее архитектруру

device = "cuda" if torch.cuda.is_available() else "cpu"

model = BertSummarizer(bert_model_name=model_name).to(device)
model


In [ ]:
# Посмотрим на генерацию без обучения

eval_data_sample = next(iter(eval_dataloader))
model.generate(
    eval_data_sample["input_ids"][:1].to(device),
    eval_data_sample["attention_mask"][:1].to(device),
    tokenizer,
)


'угле ##стеров ##ряз ##drop мнения прямои ##ru распах связа опове ##рогом необходимость подслу уходила коробка смарт шпион ##оцени реитинг учебник подош проидет 📝 правом англииском известные body подумали регла швеицар ##% ##ить шпион ##оцени реитинг помимо club12 ##ольше ##гант очертания ##лли ##зали собравшихся пошу группо мощныи хостинг удивлением настоящии υ'

## Обучение модели (1 балл)

0.25 балла за простейший рабочий цикл; 

0.5 балла за графики для лосса и метрик на трейне и валидации.

0.25 балла за логгинг в тензорборд или WandB

В данном разделе вам необходимо **реализовать цикл для обучения модели**


In [ ]:
# Пример обучения на одной итерации
# Все помнят, что надо предсказывать следующий токен?


def train_step(
    model, input_ids, attention_mask, decoder_input_ids, optimizer, criterion
):
    model.train()
    optimizer.zero_grad()
    outputs = model(input_ids, attention_mask, decoder_input_ids)
    loss = criterion(outputs.view(-1, outputs.size(-1)), decoder_input_ids.view(-1))
    loss.backward()
    optimizer.step()

    return loss.item()

## Метрики качества (1 балл)

По 0.33 балла за измерение каждой из предлагаемых метрик

**Реализуйте функицию для подсчета метрик качества суммаризации.**

Что мы хотим считать:
 1. [HuggingFace Rouge](https://huggingface.co/spaces/evaluate-metric/rouge)
 2. [HuggingFace Bleu](https://huggingface.co/spaces/evaluate-metric/bleu)
 3. [HuggingFace BERT Score](https://huggingface.co/spaces/evaluate-metric/bertscore)

In [ ]:
import torch
import evaluate
import numpy as np

rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
bertscore = evaluate.load("bertscore")


def compute_metrics(preds, refs, use_bertscore=True):
    r = rouge.compute(predictions=preds, references=refs)
    b = bleu.compute(predictions=preds, references=[[x] for x in refs])

    out = {
        "rouge1": r["rouge1"],
        "rouge2": r["rouge2"],
        "rougeL": r["rougeL"],
        "bleu": b["bleu"],
    }

    if use_bertscore:
        bs = bertscore.compute(predictions=preds, references=refs, lang="ru")
        out["bertscore_f1"] = float(np.mean(bs["f1"]))

    return out


def evaluation(model, dataloader, tokenizer, max_batches=5, strategy="greedy", use_bertscore=True):
    model.eval()
    device = next(model.parameters()).device

    preds, refs = [], []

    for i, batch in enumerate(dataloader):
        if i >= max_batches:
            break

        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)

        for j in range(ids.size(0)):
            pred = model.generate(ids[j : j + 1], mask[j : j + 1], tokenizer, strategy=strategy)
            preds.append(pred)

            lab = batch["labels"][j].tolist()
            lab = [x for x in lab if x != -100]
            refs.append(tokenizer.decode(lab, skip_special_tokens=True))

    return compute_metrics(preds, refs, use_bertscore=use_bertscore)


## Обучение модели (0.5 балла)
**Обучите модель, сохраните лучшую версию** (метод `.save_pretrained()` объекта класса AutoModel... или `torch.save()`) **и добавьте пример генерации**. Учтите, что если изменялся токенизатор (а лучше просто по умолчанию), его тоже нужно сохранить.

Для сравнения оценки качества генерации по значениям реализованных метрик можете запустить ruT5-small без дообучения. Мы намеренно даем бейзлайн именно в таком виде.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# --- обучение нашей модели ---
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)
criterion = nn.CrossEntropyLoss(ignore_index=-100)

train_losses = []
val_rougeL = []
best_score = -1

epochs = 1
max_steps = 200  # чтобы не ждать вечность, можно увеличить

for ep in range(epochs):
    for step, batch in enumerate(train_dataloader):
        if step >= max_steps:
            break

        model.train()
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        decoder_input_ids = batch["decoder_input_ids"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids, attention_mask, decoder_input_ids)
        loss = criterion(logits.view(-1, logits.size(-1)), labels.view(-1))

        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    # быстрая валидация (без BERTScore, он медленный)
    m = evaluation(model, eval_dataloader, tokenizer, max_batches=2, use_bertscore=False)
    val_rougeL.append(m["rougeL"])

    if m["rougeL"] > best_score:
        best_score = m["rougeL"]
        torch.save(model.state_dict(), "bert_summarizer_best.pt")
        tokenizer.save_pretrained("bert_summarizer_tokenizer")

print("best rougeL:", best_score)

plt.plot(train_losses)
plt.title("train loss")
plt.show()

plt.plot(val_rougeL)
plt.title("val rougeL")
plt.show()

# пример генерации
batch = next(iter(eval_dataloader))
print(model.generate(batch["input_ids"][:1].to(device), batch["attention_mask"][:1].to(device), tokenizer, strategy="beam"))

# --- бейзлайн: ruT5-small (без дообучения) ---
base_name = "cointegrated/rut5-small"  # можно заменить на любую seq2seq модель
tok2 = AutoTokenizer.from_pretrained(base_name)
m2 = AutoModelForSeq2SeqLM.from_pretrained(base_name)

text = dataset[0]["text"]
inp = tok2("summarize: " + text, return_tensors="pt", max_length=512, truncation=True)
out_ids = m2.generate(**inp, max_new_tokens=64)
summary = tok2.decode(out_ids[0], skip_special_tokens=True)

summary


## Реализация менее жадных стратегий выбора следующего токена (4 балла)
Всегда ли выбор наиболее вероятного токена на каждом шаге – это лучшая стратегия для генерации текста?

<details>
    <summary>Спойлер</summary>
    <p>Нет</p>
</details>

**Сравнение стратегий для генерации текста:**

| Strategy | Description | Pros & Cons |
| --- | --- | --- |
| Greedy Search | Chooses the word with the highest probability as the next word in the sequence. | **Pros:** Simple and fast. <br><br/> **Cons:** Can lead to repetitive and incoherent text. |
| Sampling with Temperature | Introduces randomness in the word selection. A higher temperature leads to more randomness. | **Pros:** Allows exploration and diverse output. <br><br/> **Cons:** Higher temperatures can lead to nonsensical outputs. |
| Nucleus Sampling (Top-p Sampling) | Selects the next word from a truncated vocabulary, the "nucleus" of words <br/> that have a cumulative probability exceeding a pre-specified threshold (p). | **Pros:** Balances diversity and quality. <br><br/> **Cons:** Setting an optimal 'p' can be tricky. |
| Beam Search | Explores multiple hypotheses (sequences of words) at each step, and keeps <br/> the 'k' most likely, where 'k' is the beam width. | **Pros:** Produces more reliable results than greedy search. <br><br/> **Cons:** Can lack diversity and lead to generic responses. |
| Top-k Sampling | Randomly selects the next word from the top 'k' words with the highest probabilities. | **Pros:** Introduces randomness, increasing output diversity. <br><br/> **Cons:** Random selection can sometimes lead to less coherent outputs. |
| Length Normalization | Prevents the model from favoring shorter sequences by dividing the log probabilities <br/> by the sequence length raised to some power. | **Pros:** Makes longer and potentially more informative sequences more likely. <br><br/> **Cons:** Tuning the normalization factor can be difficult. |
| Stochastic Beam Search | Introduces randomness into the selection process of the 'k' hypotheses in beam search. | **Pros:** Increases diversity in the generated text. <br><br/> **Cons:** The trade-off between diversity and quality can be tricky to manage. |
| Decoding with Minimum Bayes Risk (MBR) | Chooses the hypothesis (out of many) that minimizes expected loss under a loss function. | **Pros:** Optimizes the output according to a specific loss function. <br><br/> **Cons:** Computationally more complex and requires a good loss function. |

Ссылки на докуметацию:
- [reference for `AutoModelForCausalLM.generate()`](https://huggingface.co/docs/transformers/v4.29.1/en/main_classes/text_generation#transformers.GenerationMixin.generate)
- [reference for `AutoTokenizer.decode()`](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.PreTrainedTokenizer.decode)
- Huggingface [docs on generation strategies](https://huggingface.co/docs/transformers/generation_strategies)

**1. Реализуйте стратегию Top-k в методе `generate`** (1 балл).   

**2. Реализуйте стратегию Nucleus Sampling (Top-p) в методе `generate`** (1 балл)

**3. Реализуйте стратегию Beam Search** (2 балла)

Получилось ли улучшить генерацию?

In [ ]:
# Сравним стратегии на одном примере + маленькая проверка метрик

batch = next(iter(eval_dataloader))
x = batch["input_ids"][:1].to(device)
m = batch["attention_mask"][:1].to(device)

print("greedy:", model.generate(x, m, tokenizer, strategy="greedy"))
print("top_k:", model.generate(x, m, tokenizer, strategy="top_k", top_k=30))
print("top_p:", model.generate(x, m, tokenizer, strategy="top_p", top_p=0.9))
print("beam:", model.generate(x, m, tokenizer, strategy="beam", num_beams=3))

res_g = evaluation(model, eval_dataloader, tokenizer, max_batches=3, strategy="greedy", use_bertscore=False)
res_b = evaluation(model, eval_dataloader, tokenizer, max_batches=3, strategy="beam", use_bertscore=False)

print("metrics greedy:", res_g)
print("metrics beam:", res_b)

print("вывод: обычно beam/top_p дают текст почище, но метрики могут меняться по-разному")


## Бонус (1 балл)

Что требуется сделать:

- от имеющейся модели "откусить" только декодерную часть
- написать цикл обучения (скорее поправить имеющийся) и дообучить декодер
- посмотреть качество генерации по метрикам и "глазами"
- ответить на вопрос "Дает ли применение Encoder-Decoder архитектуры значительный буст в качестве генерации?" с пруфами

In [ ]:
# Бонус (по желанию): дообучаем только декодер (замораживаем BERT)

do_bonus = False

if do_bonus:
    before = evaluation(model, eval_dataloader, tokenizer, max_batches=3, use_bertscore=False)
    print("до:", before)

    for p in model.bert.parameters():
        p.requires_grad = False

    opt = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
    crit = nn.CrossEntropyLoss(ignore_index=-100)

    for step, batch in enumerate(train_dataloader):
        if step >= 100:
            break

        model.train()
        opt.zero_grad()

        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        dec = batch["decoder_input_ids"].to(device)
        lab = batch["labels"].to(device)

        logits = model(ids, mask, dec)
        loss = crit(logits.view(-1, logits.size(-1)), lab.view(-1))

        loss.backward()
        opt.step()

    after = evaluation(model, eval_dataloader, tokenizer, max_batches=3, use_bertscore=False)
    print("после:", after)

    print("вывод: encoder-decoder обычно лучше, если энкодер учится, но даже дообучение декодера может чуть помочь")
